In [3]:
# Projet portfolio - E-commerce Olist
# Etape 1 : chargement des tables brutes + jointures en un dataframe analytique unique

# Tables sources (CSV Kaggle "Brazilian E-Commerce Public Dataset by Olist") :
  
import pandas as pd

# 1. Chargement des CSV bruts

orders = pd.read_csv("olist_orders_dataset.csv", parse_dates=[
    "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date"
])
customers = pd.read_csv("olist_customers_dataset.csv")
order_items = pd.read_csv("olist_order_items_dataset.csv")
products = pd.read_csv("olist_products_dataset.csv")
payments = pd.read_csv("olist_order_payments_dataset.csv")
reviews = pd.read_csv("olist_order_reviews_dataset.csv")
sellers = pd.read_csv("olist_sellers_dataset.csv")
category_translation = pd.read_csv("product_category_name_translation.csv")


In [4]:
# 2. Nettoyage rapide avant jointure
# ----------------------------------------------------------------------
# Traduction des catégories produit (PT -> EN) pour lisibilité
products = products.merge(category_translation, on="product_category_name", how="left")

# Paiements : une commande peut avoir plusieurs lignes de paiement (plusieurs moyens)
# -> on agrège au niveau commande (somme de la valeur payée, nb de moyens de paiement)
payments_agg = (
    payments.groupby("order_id")
    .agg(
        payment_value=("payment_value", "sum"),
        payment_installments_max=("payment_installments", "max"),
        payment_type_principal=("payment_type", lambda x: x.mode().iloc[0] if not x.mode().empty else None),
    )
    .reset_index()
)

# Avis : une commande peut avoir plusieurs avis (rare) -> on garde le plus récent
reviews_dedup = (
    reviews.sort_values("review_creation_date")
    .drop_duplicates(subset="order_id", keep="last")
    [["order_id", "review_score", "review_comment_title", "review_comment_message"]]
)

# Order items : une commande peut avoir plusieurs lignes (plusieurs produits)
# -> on agrège au niveau commande pour la jointure principale
order_items_agg = (
    order_items.groupby("order_id")
    .agg(
        nb_produits=("order_item_id", "count"),
        prix_total=("price", "sum"),
        frais_port_total=("freight_value", "sum"),
    )
    .reset_index()
)

In [5]:
# 3. Jointures successives (equivalent des JOIN SQL)
# ----------------------------------------------------------------------
df = orders.merge(customers, on="customer_id", how="left")
df = df.merge(order_items_agg, on="order_id", how="left")
df = df.merge(payments_agg, on="order_id", how="left")
df = df.merge(reviews_dedup, on="order_id", how="left")

# ----------------------------------------------------------------------
# 4. Colonnes calculées utiles pour l'analyse business
# ----------------------------------------------------------------------
df["delai_livraison_jours"] = (
    df["order_delivered_customer_date"] - df["order_purchase_timestamp"]
).dt.days

df["retard_estime_jours"] = (
    df["order_delivered_customer_date"] - df["order_estimated_delivery_date"]
).dt.days  # positif = livré en retard par rapport à l'estimation

In [6]:
# 6. Export intermédiaire pour la suite du projet (EDA, analyse business)
df.to_csv("olist_dataset_analytique.csv", index=False)